# ErgoVision — Ergonomic Risk Assessment

Vision-based ergonomic risk assessment using YOLOv8-pose + RULA-inspired scoring.

**Esecuzione locale**: i dati vanno in una cartella dentro `data/`.
La pipeline gestisce automaticamente sia immagini che video.

### Output
Tutti i risultati vengono salvati in `outputs/<dataset_name>/`:
- CSV dettagliati (frame_person_results.csv, video_summary.csv)
- Frame annotati con skeleton + risk overlay
- Video annotati (se input video)
- Grafici matplotlib (distribuzione rischio, istogrammi angoli, ...)
- Esempi LOW / MEDIUM / HIGH risk
- Report Markdown sperimentale

## 1. Installazione dipendenze

In [1]:
!pip install -q ultralytics opencv-python-headless numpy matplotlib

## 2. Import e configurazione

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, '.')

import ergovision
from ergovision import ErgoPipeline, ExperimentConfig
from ergovision.dataset import inspect_dataset
from ergovision.config import IMAGE_EXTENSIONS, VIDEO_EXTENSIONS

print(f'ErgoVision v{ergovision.__version__} loaded')

ErgoVision v0.5.0 loaded


## 3. Dataset

Crea una cartella dentro `data/` con il tuo dataset, tipo:
```
data/
  └── nome-dataset/       <-- crei tu questa
        ├── video1.mp4
        └── ...
```
Poi imposta `DATASET_NAME = "nome-dataset"` qui sotto.

In [3]:
# ============================================================
# SCEGLI IL NOME DELLA TUA CARTELLA DENTRO data/
# ============================================================
DATASET_NAME = 'mp4'   # <-- CAMBIA QUESTO
# ============================================================

DATA_PATH = Path('data') / DATASET_NAME

if not DATA_PATH.exists():
    print(f'Cartella non trovata: {DATA_PATH}')
    print('\nCartelle disponibili in data/:')
    root = Path('data')
    if root.exists():
        for d in sorted(root.iterdir()):
            if d.is_dir():
                print(f'  - {d.name}/')
    else:
        root.mkdir(parents=True, exist_ok=True)
        print('  (nessuna -- creane una e mettici i file)')
    raise FileNotFoundError(f'Crea data/{DATASET_NAME}/ e riprova')

print(f'Dataset: {DATA_PATH.resolve()}')

info = inspect_dataset(DATA_PATH)
print(f'  Video:    {info.n_videos}')
print(f'  Immagini: {info.n_images}')

if info.total_files == 0:
    print(f'\nATTENZIONE: {DATA_PATH} vuota! Metti file e riavvia.')

Dataset: C:\Users\MarioBove\Desktop\ErgoVision\data\mp4
  Video:    3
  Immagini: 0


## 4. Esecuzione pipeline (sperimentale)

Usa la nuova pipeline sperimentale che:
- Se ci sono video: estrae i frame automaticamente
- Se ci sono immagini: le processa direttamente
- Produce output completi in `outputs/<dataset_name>/`

In [4]:
# Configurazione
from ergovision.config import ExperimentConfig

config = ExperimentConfig(dataset_name=DATASET_NAME)
config.frame_sampling_fps = 1.0
config.max_frames_per_video = 100
config.save_annotated_frames = True
config.save_annotated_video = True
config.save_failure_cases = True
config.save_csv = True
config.save_plots = True

print(f'Config: frame_sampling={config.frame_sampling_fps}fps, '
      f'max_frames_per_video={config.max_frames_per_video}')

# Esecuzione
pipeline = ErgoPipeline(config=config)
result = pipeline.run(dataset_name=DATASET_NAME, verbose=True)

Config: frame_sampling=1.0fps, max_frames_per_video=100
  ErgoVision — Experimental Pipeline  |  Dataset: mp4

[1/6] Inspecting dataset: data\mp4
  mp4: 3 video(s)

[2/6] Extracting frames from 3 video(s)...
  WS10-SN22106779-HD720dt__23_04_2024_13_53_19_4.0m_cropped_L_view.mp4: 100 frames
  WS20-SN25431032-HD720dt__07_05_2024_10_42_02_5.0m_L_view.mp4: 100 frames
  WS30-SN22354094-HD720dt__09_05_2024_09_50_25_5.0m_L_view.mp4: 100 frames
  Total extracted frames: 300

[3/6] Two-stage pipeline: detector=yolov8l.pt pose=yolov8l-pose.pt ...
  [15/300]  +1 valid
  [30/300]  +1 valid
  [45/300]  +1 valid
  [60/300]  +1 valid
  [75/300]  +1 valid
  [90/300]  +1 valid
  [105/300]  +2 valid
  [120/300]  +1 valid
  [135/300]  +1 valid
  [150/300]  +1 valid
  [165/300]  +1 valid
  [180/300]  +1 valid
  [195/300]  +0 valid
  [210/300]  +0 valid
  [225/300]  +0 valid
  [240/300]  +0 valid
  [255/300]  +0 valid
  [270/300]  +1 valid
  [285/300]  +1 valid
  [300/300]  +1 valid

[4/6] Computing ergono

NameError: name 'json' is not defined

## 5. Preview risultati

In [5]:
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
from collections import Counter

OUTPUT_DIR = Path('outputs') / DATASET_NAME
csv_path = OUTPUT_DIR / 'csv' / 'frame_person_results.csv'

if csv_path.exists():
    import csv
    with open(csv_path, encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    print(f'Righe totali: {len(rows)}')

    if rows:
        # Tabella riassuntiva
        print(f'{"frame_id":25s} {"risk_level":8s} {"score":5s} {"trunk":>6s} {"neck":>6s} {"upper_arm":>8s}')
        print('-' * 60)
        for r in rows[:10]:
            fid = r.get('frame_id', '?')[:24]
            print(f'{fid:25s} {r.get("risk_level", "?"):8s} '
                  f'{r.get("risk_score", "?"):5s} '
                  f'{r.get("trunk_angle", "?"):>6s} '
                  f'{r.get("neck_angle", "?"):>6s} '
                  f'{r.get("upper_arm_angle_left", "?"):>8s}')
        if len(rows) > 10:
            print(f'... e {len(rows) - 10} righe in piu')

    # Distribuzione rischio
    classes = [r['risk_level'] for r in rows]
    counts = Counter(classes)
    
    fig, ax = plt.subplots(figsize=(6, 4))
    colors = {'LOW': '#2ecc71', 'MEDIUM': '#f1c40f', 'HIGH': '#e74c3c'}
    levels = ['LOW', 'MEDIUM', 'HIGH']
    values = [counts.get(l, 0) for l in levels]
    bar_colors = [colors.get(l, '#888') for l in levels]
    bars = ax.bar(levels, values, color=bar_colors, edgecolor='gray')
    ax.set_ylabel('Rilevazioni')
    ax.set_title('Distribuzione Rischio Ergonomico')
    for bar, cnt in zip(bars, values):
        if cnt > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                    str(cnt), ha='center', va='bottom', fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Anteprima frame annotati
    ann_dir = OUTPUT_DIR / 'annotated_frames'
    if ann_dir.exists():
        viz_files = sorted(ann_dir.glob('*'))[:6]
        if viz_files:
            fig, axes = plt.subplots(2, 3, figsize=(15, 10))
            for ax_i, vf in zip(axes.flat, viz_files):
                img = cv2.cvtColor(cv2.imread(str(vf)), cv2.COLOR_BGR2RGB)
                ax_i.imshow(img)
                ax_i.set_title(vf.name, fontsize=9)
                ax_i.axis('off')
            plt.tight_layout()
            plt.show()

    # Report
    report_path = OUTPUT_DIR / 'reports' / 'experimental_report.md'
    if report_path.exists():
        print(f'\nReport generato: {report_path}')
        print(f'Preview (prime 20 righe):')
        with open(report_path, encoding='utf-8') as f:
            for i, line in enumerate(f):
                if i < 20:
                    print(line.rstrip())
else:
    print('Nessun risultato. Esegui la pipeline prima.')

Righe totali: 100
frame_id                  risk_level score  trunk   neck upper_arm
------------------------------------------------------------
frame_000000              AL1 (LOW) 1        4.2   43.9      4.0
frame_000030              AL1 (LOW) 1       25.3   25.2     38.8
frame_000060              AL1 (LOW) 1       13.1   37.6     35.2
frame_000090              AL1 (LOW) 1        4.9   39.8     18.7
frame_000120              AL1 (LOW) 1       12.5   36.5     36.3
frame_000150              AL1 (LOW) 1       18.4   38.5     13.7
frame_000180              AL1 (LOW) 1       17.5   40.3     14.3
frame_000210              AL1 (LOW) 1       18.5   39.2     14.1
frame_000240              AL1 (LOW) 1       11.6   58.1      7.6
frame_000270              AL1 (LOW) 1       18.0   52.1      3.4
... e 90 righe in piu


C:\Users\MarioBove\AppData\Local\Temp\ipykernel_10192\4187501461.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Report generato: outputs\mp4\reports\experimental_report.md
Preview (prime 20 righe):
# Experimental Report

**Pipeline:** ErgoVision v0.2.0
**Date:** 2026-05-29 11:36
**Dataset:** `mp4`

## Dataset

The dataset `mp4` contains **1 video(s)** processed at **1.0 FPS** (max 100 frames per video).

- Frame sampling rate: 1.0 FPS
- Max frames / video: 100
- Detection confidence threshold: 0.5
- Keypoint confidence threshold: 0.3
- Total frames processed: 100

### Per-Video Processing Summary

| Video ID | Frames Processed | Frames with People | Valid Postures | Low | Med | High | Mean Risk |
|----------|-----------------|-------------------|---------------|-----|-----|------|----------|


C:\Users\MarioBove\AppData\Local\Temp\ipykernel_10192\4187501461.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
Tutti gli output sono in `outputs/<dataset_name>/`

```
outputs/<dataset_name>/
    frames/
    annotated_frames/
    annotated_videos/
    csv/
    plots/
    examples/
    reports/
```